# Práctica 3: Predicción de deserción estudiantil

## Instrucciones de la práctica

La tercera práctica consiste en construir un clasificador que prediga, al cierre del primer semestre, la situación que tendrá un estudiante al final de la duración normal de su carrera: graduado, desertor o todavía inscrito. Los datos son reales: 4424 estudiantes de licenciatura del Instituto Politécnico de Portalegre, en Portugal, que ingresaron entre los ciclos 2008/09 y 2018/19.



Los datos están en la carpeta datos/ del repositorio del curso:

desercion-entrenamiento.csv: 3539 estudiantes con sus 36 atributos y la variable objetivo Target.

desercion-prueba.csv: 885 estudiantes sin Target. Solo trae los atributos que se conocen al cierre del primer semestre; los del segundo semestre aparecen únicamente en el entrenamiento.

La descripción de cada atributo y el significado de sus códigos están en la ficha del dataset en UCI (https://archive.ics.uci.edu/dataset/697). Varios atributos nominales vienen codificados como números enteros.



Se compararán únicamente los clasificadores vistos en la unidad, llevando cada uno a su mejor versión: árbol de decisión, Naive Bayes, k vecinos más cercanos (kNN), máquinas de soporte vectorial (SVM) y regresión logística. No se aceptan ensambles ni redes neuronales.



El notebook deberá incluir:

Una descripción breve del problema y de los datos: la distribución de las clases, el tipo de cada atributo y la información disponible en el momento de decidir.

Para cada clasificador, el preprocesamiento que su modelo requiere (codificación de los atributos nominales, escalamiento, discretización), dentro de un Pipeline y con su justificación.

La optimización de los hiperparámetros principales de cada clasificador con validación cruzada estratificada sobre el conjunto de entrenamiento, con los mismos pliegues para los cinco. Como mínimo: el criterio, la profundidad y la poda del árbol; la variante y el suavizado de Naive Bayes; k, la ponderación y la distancia de kNN; el kernel, C y gamma del SVM; C y el tipo de regularización de la regresión logística, y los pesos por clase (class_weight) en los tres clasificadores que los admiten: el árbol, el SVM y la regresión logística. Puede hacerse con búsqueda en malla, aleatoria o bayesiana (e.g., Optuna), declarando el espacio de búsqueda y el número de configuraciones evaluadas, que no será menor a 50 por clasificador.

Para cada clasificador, una gráfica que muestre que la búsqueda llegó a una meseta: el F1 macro de validación contra su hiperparámetro más influyente o, en búsquedas aleatorias o bayesianas, el mejor F1 macro alcanzado contra el número de configuraciones evaluadas.

La comparación de los cinco clasificadores (optimizados) con el F1 macro como métrica principal, con la media y la desviación estándar entre pliegues, la sensibilidad y la especificidad por clase, y el tiempo de la búsqueda.

La matriz de confusión del mejor clasificador, construida con predicciones de validación cruzada, y un análisis de la clase desertor: con qué clase se confunde y por qué. 

Una estimación, con el conjunto de entrenamiento, de cuánto mejoraría el mejor clasificador si se esperara al cierre del segundo semestre, y una discusión sobre si conviene esperar.

El clasificador final (el mejor de todos), entrenado con todo el conjunto de entrenamiento y obtener las predicciones sobre el conjunto de prueba.

Una discusión y conclusiones sobre el desempeño de cada clasificador en este problema.

Además del notebook, se entregará un archivo CSV generado por el propio notebook con las predicciones del clasificador final para los 885 estudiantes de prueba, con exactamente dos columnas: id y Target, cuyos valores serán dropout, enrolled, o graduate. Los profesores calcularán el F1 macro de prueba de cada entrega. Además de la calificación de la práctica, se otorgarán puntos adicionales a la calificación final del curso, de la siguiente forma: 3, 2 y 1 puntos para el primer, segundo y tercer lugar, respectivamente.



El notebook deberá ejecutarse completo y sin errores desde el inicio (Restart & Run All), utilizar rutas relativas, fijar la semilla de aleatoriedad cuando aplique, y declarar las versiones de las librerías utilizadas. El repositorio deberá incluir un README con la descripción del dataset, los requisitos de ejecución, y las instrucciones de uso. Se entregará el enlace al repositorio, el notebook y el archivo de predicciones en la plataforma del curso.

## 0. Librerías y semilla

In [ ]:
import os, time, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.stats import loguniform, randint
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, KBinsDiscretizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)

print('numpy', np.__version__)
print('pandas', pd.__version__)
print('scipy', scipy.__version__)
print('matplotlib', matplotlib.__version__)
print('scikit-learn', sklearn.__version__)

## 1. Descripción del problema y de los datos

El problema es predecir, al cierre del primer semestre, qué pasará con un estudiante al final de la duración normal de su carrera: `Graduate`, `Dropout` o `Enrolled`. Son 3539 estudiantes en el entrenamiento y 885 en la prueba, del Instituto Politécnico de Portalegre.

Las clases están desbalanceadas: casi la mitad son graduados, un tercio desertores y solo 18% siguen inscritos. Por eso la métrica principal es el F1 macro, que da el mismo peso a las tres clases.

In [ ]:
train = pd.read_csv('desercion-entrenamiento.csv')
test = pd.read_csv('desercion-prueba.csv')
print('entrenamiento:', train.shape)
print('prueba:', test.shape)

# distribución de las clases
conteo = train['Target'].value_counts()
print(pd.DataFrame({'estudiantes': conteo, 'porcentaje': (conteo / len(train) * 100).round(1)}))

conteo.plot(kind='bar', color='steelblue')
plt.title('Distribución de Target (entrenamiento)')
plt.ylabel('estudiantes')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# tipo de cada atributo
nominales = ['Marital status', 'Application mode', 'Course', 'Previous qualification', 'Nacionality',
             "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"]
binarios = ['Daytime/evening attendance', 'Displaced', 'Educational special needs', 'Debtor',
            'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']
sem2 = [c for c in train.columns if '2nd sem' in c]

# X: solo lo que se conoce al cierre del primer semestre (lo que también trae la prueba)
# X2: todos los atributos, incluidos los del segundo semestre (solo se usa en la sección 7)
y = train['Target']
X2 = train.drop(columns=['id', 'Target'])
X = X2.drop(columns=sem2)
X_prueba = test[X.columns]

tipo = []
for c in X2.columns:
    if c in nominales:
        tipo.append('nominal')
    elif c in binarios:
        tipo.append('binario')
    else:
        tipo.append('numérico')

tipos = pd.DataFrame({'atributo': X2.columns,
                      'tipo': tipo,
                      'valores distintos': [X2[c].nunique() for c in X2.columns],
                      'disponible al cierre del 1er semestre': [c in X.columns for c in X2.columns]})
tipos

Al momento de decidir (cierre del primer semestre) solo se conocen los datos del ingreso (demográficos, socioeconómicos, de admisión), las materias del primer semestre y los indicadores macroeconómicos. Los atributos del segundo semestre (`2nd sem`) solo existen en el entrenamiento, así que los modelos no los usan; se usan únicamente en la sección 7 para ver cuánto mejoraría esperar.

El atributo `id` no se usa para entrenar.

## 2. Preprocesamiento por clasificador

- **Árbol de decisión:** no necesita escalar. Los nominales se codifican con `OrdinalEncoder` para no generar tantas columnas.
- **Naive Bayes:** los nominales van con one-hot. Para la variante gaussiana los numéricos quedan sin escalar. Para las variantes multinomial, complemento y Bernoulli se discretizan los numéricos en intervalos (`KBinsDiscretizer`), porque esas variantes trabajan con conteos o valores binarios.
- **kNN, SVM y regresión logística:** one-hot para los nominales y `StandardScaler` para el resto. Los tres dependen de la escala (distancias o productos punto), así que sin escalar un atributo con valores grandes dominaría.
- En el one-hot las categorías con menos de 10 estudiantes se agrupan en una sola, y las que no aparecen en entrenamiento se mandan a ese grupo.
- Todo está dentro de un `Pipeline`, así el escalamiento y la codificación se ajustan solo con los datos de entrenamiento de cada pliegue.

In [ ]:
# los nominales (códigos enteros) se tratan distinto según el modelo;
# el resto de columnas (numéricas y binarias) entra por "remainder"
uno_hot = OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10, sparse_output=False)

# árbol: sin escalar, nominales con código ordinal
prep_arbol = ColumnTransformer([('nom', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), nominales)],
                               remainder='passthrough')

# kNN, SVM y regresión logística: one-hot + escalar
prep_esc = ColumnTransformer([('nom', uno_hot, nominales)], remainder=StandardScaler())

# Naive Bayes gaussiano: one-hot, sin escalar
prep_gauss = ColumnTransformer([('nom', uno_hot, nominales)], remainder='passthrough')

# Naive Bayes discretos: one-hot + discretizar los numéricos en intervalos
prep_disc = ColumnTransformer([('nom', uno_hot, nominales)],
                              remainder=KBinsDiscretizer(n_bins=5, encode='onehot-dense', strategy='quantile'))

pipe_arbol = Pipeline([('prep', prep_arbol), ('clf', DecisionTreeClassifier(random_state=SEED))])
pipe_nb = Pipeline([('prep', prep_gauss), ('clf', GaussianNB())])
pipe_knn = Pipeline([('prep', prep_esc), ('clf', KNeighborsClassifier())])
pipe_svm = Pipeline([('prep', prep_esc), ('clf', SVC(random_state=SEED, cache_size=500))])
pipe_lr = Pipeline([('prep', prep_esc), ('clf', LogisticRegression(solver='saga', max_iter=3000, random_state=SEED))])

## 3. Optimización de hiperparámetros

Se usó búsqueda aleatoria (`RandomizedSearchCV`) con F1 macro como métrica, 60 configuraciones por clasificador y los mismos 5 pliegues estratificados para todos. Los espacios de búsqueda están en el código de abajo.

In [ ]:
# mismos pliegues para los cinco clasificadores
pliegues = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# --- espacios de búsqueda ---

# árbol
esp_arbol = {
    'clf__criterion': ['gini', 'entropy', 'log_loss'],
    'clf__max_depth': [3, 4, 5, 6, 8, 10, 12, 15, 20, None],
    'clf__ccp_alpha': loguniform(1e-4, 5e-2),          # poda
    'clf__class_weight': [None, 'balanced'],
}

# Naive Bayes: cada diccionario es una variante
esp_nb = [
    {'clf': [GaussianNB()], 'clf__var_smoothing': loguniform(1e-11, 1e-1)},
    {'prep': [prep_disc], 'prep__remainder__n_bins': [3, 5, 8, 10],
     'clf': [MultinomialNB()], 'clf__alpha': loguniform(1e-2, 10)},
    {'prep': [prep_disc], 'prep__remainder__n_bins': [3, 5, 8, 10],
     'clf': [ComplementNB()], 'clf__alpha': loguniform(1e-2, 10)},
    {'prep': [prep_disc], 'prep__remainder__n_bins': [3, 5, 8, 10],
     'clf': [BernoulliNB()], 'clf__alpha': loguniform(1e-2, 10)},
]

# kNN
esp_knn = {
    'clf__n_neighbors': randint(1, 81),
    'clf__weights': ['uniform', 'distance'],
    'clf__metric': ['euclidean', 'manhattan', 'chebyshev'],
}

# SVM
esp_svm = {
    'clf__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'clf__C': loguniform(1e-2, 1e2),
    'clf__gamma': loguniform(1e-4, 1e-1),
    'clf__degree': [2, 3],
    'clf__class_weight': [None, 'balanced'],
}

# regresión logística (l1_ratio: 0 = L2, 1 = L1, en medio = elastic net)
esp_lr = {
    'clf__C': loguniform(1e-3, 1e2),
    'clf__l1_ratio': [0, 0.25, 0.5, 0.75, 1],
    'clf__class_weight': [None, 'balanced'],
}

n_config = 60   # configuraciones evaluadas por clasificador (búsqueda aleatoria)

modelos = {
    'Árbol de decisión': (pipe_arbol, esp_arbol),
    'Naive Bayes': (pipe_nb, esp_nb),
    'kNN': (pipe_knn, esp_knn),
    'SVM': (pipe_svm, esp_svm),
    'Regresión logística': (pipe_lr, esp_lr),
}

In [ ]:
busquedas = {}
tiempos = {}

for nombre, (pipe, espacio) in modelos.items():
    inicio = time.time()
    busqueda = RandomizedSearchCV(pipe, espacio, n_iter=n_config, scoring='f1_macro',
                                  cv=pliegues, n_jobs=-1, random_state=SEED)
    busqueda.fit(X, y)
    tiempos[nombre] = time.time() - inicio
    busquedas[nombre] = busqueda
    print(nombre, '| mejor F1 macro:', round(busqueda.best_score_, 4), '| tiempo:', round(tiempos[nombre]), 's')
    print('   ', {k: v for k, v in busqueda.best_params_.items() if k not in ('prep', 'clf')},
          '| clasificador:', type(busqueda.best_params_.get('clf', busqueda.best_estimator_[-1])).__name__)

## 4. Gráficas de la búsqueda (meseta)

Para cada clasificador se grafica el F1 macro de cada configuración y el mejor F1 macro alcanzado hasta ese momento.

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(15, 8))
ejes = ejes.ravel()

for ax, (nombre, busqueda) in zip(ejes, busquedas.items()):
    f1 = busqueda.cv_results_['mean_test_score']
    n = np.arange(1, len(f1) + 1)
    ax.scatter(n, f1, s=10, alpha=0.4, label='cada configuración')
    ax.plot(n, np.maximum.accumulate(f1), color='red', label='mejor hasta ahora')
    ax.set_title(nombre)
    ax.set_xlabel('configuraciones evaluadas')
    ax.set_ylabel('F1 macro (validación)')
    ax.legend()

ejes[5].axis('off')
plt.tight_layout()
plt.show()

En los cinco la línea roja se queda plana mucho antes de llegar a las 60 configuraciones, así que más búsqueda no cambiaría mucho el resultado. El árbol, Naive Bayes y la regresión logística llegan a su mejor valor en las primeras 10 a 15 configuraciones; kNN y SVM tardan un poco más (alrededor de 20 y 40).

## 5. Comparación de los clasificadores

In [ ]:
clases = sorted(y.unique())
predicciones = {}   # predicciones de validación cruzada de cada modelo
filas = []

for nombre, busqueda in busquedas.items():
    pred = cross_val_predict(busqueda.best_estimator_, X, y, cv=pliegues, n_jobs=-1)
    predicciones[nombre] = pred
    cm = confusion_matrix(y, pred, labels=clases)

    fila = {'Clasificador': nombre,
            'F1 macro (media)': busqueda.cv_results_['mean_test_score'][busqueda.best_index_],
            'F1 macro (desv. est.)': busqueda.cv_results_['std_test_score'][busqueda.best_index_]}

    for i, c in enumerate(clases):
        tp = cm[i, i]
        fn = cm[i].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        fila['Sensibilidad ' + c] = tp / (tp + fn)
        fila['Especificidad ' + c] = tn / (tn + fp)

    fila['Tiempo búsqueda (s)'] = tiempos[nombre]
    filas.append(fila)

tabla = pd.DataFrame(filas).set_index('Clasificador')
tabla.round(3)

La regresión logística y el SVM son los mejores y quedan casi empatados (la diferencia es menor que la desviación estándar entre pliegues). Después viene el árbol, luego Naive Bayes y al final kNN.

En todos los modelos la clase `Enrolled` es la más difícil (sensibilidad baja) y `Graduate` la más fácil. La regresión logística fue la más lenta en la búsqueda, y el árbol y Naive Bayes los más rápidos.

## 6. Matriz de confusión del mejor clasificador

In [ ]:
mejor = tabla['F1 macro (media)'].idxmax()
print('Mejor clasificador:', mejor)

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y, predicciones[mejor], labels=clases, cmap='Blues', ax=ejes[0])
ejes[0].set_title('Conteos')
ConfusionMatrixDisplay.from_predictions(y, predicciones[mejor], labels=clases, normalize='true',
                                        values_format='.2f', cmap='Blues', ax=ejes[1])
ejes[1].set_title('Normalizada por fila (clase real)')
plt.tight_layout()
plt.show()

# a dónde se van los desertores
cm = confusion_matrix(y, predicciones[mejor], labels=clases)
print(pd.Series(cm[0] / cm[0].sum(), index=clases).round(3))

El mejor clasificador detecta como desertor a cerca de 7 de cada 10 desertores. Los que falla los manda sobre todo a `Enrolled` (20%) y menos a `Graduate` (11%).

Esto pasa porque los desertores y los que siguen inscritos se parecen al final del primer semestre: ambos grupos tienen materias aprobadas parecidas y todavía no se sabe si el estudiante va a abandonar o solo va más lento. Además, `Enrolled` es una clase pequeña y mezclada, y parte de la deserción ocurre después del primer semestre, cuando todavía no hay señales.

## 7. ¿Conviene esperar al segundo semestre?

In [ ]:
# mismo mejor modelo y mismos hiperparámetros, pero con todos los atributos (X2) y los mismos pliegues
mejor_modelo = busquedas[mejor].best_estimator_

f1_sem1 = cross_val_score(mejor_modelo, X, y, cv=pliegues, scoring='f1_macro', n_jobs=-1)
f1_sem2 = cross_val_score(mejor_modelo, X2, y, cv=pliegues, scoring='f1_macro', n_jobs=-1)

pred2 = cross_val_predict(mejor_modelo, X2, y, cv=pliegues, n_jobs=-1)
sens_sem1 = confusion_matrix(y, predicciones[mejor], labels=clases).diagonal() / y.value_counts()[clases].values
sens_sem2 = confusion_matrix(y, pred2, labels=clases).diagonal() / y.value_counts()[clases].values

comparacion = pd.DataFrame({'F1 macro (media)': [f1_sem1.mean(), f1_sem2.mean()],
                            'F1 macro (desv. est.)': [f1_sem1.std(), f1_sem2.std()]},
                           index=['Cierre 1er semestre', 'Cierre 2do semestre'])
for i, c in enumerate(clases):
    comparacion['Sensibilidad ' + c] = [sens_sem1[i], sens_sem2[i]]
comparacion.round(3)

Con los datos del segundo semestre el F1 macro sube de 0.68 a 0.72 (unos 4 puntos), y la sensibilidad de `Dropout` sube de 0.69 a 0.73. La mejora existe pero es pequeña.

Esperar implica perder un semestre en el que se podría apoyar al estudiante. Como la ganancia es modesta, no conviene esperar: es mejor decidir al cierre del primer semestre y aprovechar el tiempo para intervenir. Esta estimación se hizo con los mismos hiperparámetros del mejor modelo, sin volver a optimizar.

## 8. Clasificador final y predicciones

In [ ]:
final = clone(busquedas[mejor].best_estimator_)
final.fit(X, y)

pred_prueba = final.predict(X_prueba)

salida = pd.DataFrame({'id': test['id'], 'Target': pd.Series(pred_prueba).str.lower()})
salida.to_csv('predicciones.csv', index=False)

print(salida.shape)
print(salida['Target'].value_counts())
salida.head()

## 9. Discusión y conclusiones

- **Regresión logística:** la mejor en F1 macro. Un modelo lineal con las clases balanceadas y regularización fue suficiente, lo que sugiere que las clases se separan de forma casi lineal con estos atributos. Su desventaja fue el tiempo de búsqueda.
- **SVM:** casi igual de bueno que la regresión logística (kernel rbf). Es más lento y no mejora, así que el kernel no lineal no aporta mucho aquí.
- **Árbol de decisión:** queda en medio. Funciona mejor con poda y profundidad baja; sin eso se sobreajusta. Es el más fácil de interpretar.
- **Naive Bayes:** está por debajo, probablemente porque los atributos están correlacionados (por ejemplo, materias inscritas y aprobadas) y el modelo supone que son independientes.
- **kNN:** el peor, sobre todo en `Enrolled`. Con tantos atributos (muchos de ellos one-hot) las distancias pierden sentido.
- Los clasificadores usaron `class_weight='balanced'` cuando se pudo, lo que ayudó con la clase `Enrolled`, que es la más difícil.
- Las medias de F1 macro de validación cruzada son un poco optimistas porque se eligieron como las mejores de la búsqueda; el resultado en la prueba puede ser algo menor.